# Belt Conveyor: geared drive — sensor vs camera

| | Sensor | Camera |
|---|---|---|
| Data | CORA T1 temperature | CORA infrared camera |
| Classes | healthy vs gear-worn | healthy vs gear-worn |
| Tested on | 16 runs, one speed hidden at a time | the same 16 runs |
| Code | `Models/belt_conveyor.py experiment` (both branches) | |

CORA is one lab motor/gearbox rig, used as a stand-in for a **geared** conveyor drive (not belts or idlers). Both branches use the same runs but are trained separately.

All of this machine's experiment code is in its `Models/` file; shared notebook set-up is in `notebook_helpers.py`. Run the cells from top to bottom with the **Capstone (.venv)** kernel.

In [1]:
from notebook_helpers import *

LOCAL = setup([
    ("Sensor: CORA T1 samples", "Processed_Data/cora_gear_wear/t1_bounded_samples.npz", "*", 1),
    ("Camera: CORA infrared stacks", "Raw_Data/cora_gear_wear", "*.mat", 16),
    ("Sensor: CORA vibration (Vx)", "Raw_Data/cora_vibration", "*_Vx.csv", 16),
])

,environment
git commit,cf5f533
python,3.11.9
kernel,project .venv
numpy,2.4.6
scipy,1.17.1
pandas,2.3.3
scikit-learn,1.9.1
opencv-python,4.14.0.94
matplotlib,3.11.2


,location,files,MB,local
data,,,,
Sensor: CORA T1 samples,Processed_Data/cora_gear_wear/t1_bounded_sampl...,1/1,1.7,True
Camera: CORA infrared stacks,Raw_Data/cora_gear_wear,16/16,390.5,True
Sensor: CORA vibration (Vx),Raw_Data/cora_vibration,28/16,3078.1,True


All data is local: running fully offline.


## Run both branches
One script trains the sensor model and the camera model on the same four rounds (each round hides one speed).

In [2]:
before = load("belt_conveyor_cora.json")
if all(LOCAL.values()):
    run_script("Models/belt_conveyor.py", "experiment", tail=1)
belt = load("belt_conveyor_cora.json")
check(before, belt, [("sensor", "model"), ("infrared", "model")], "Belt Conveyor")
show({"Sensor: T1 temperature": belt["sensor"]["model"],
      "Camera: infrared": belt["infrared"]["model"],
      "Always guess 'worn'": belt["sensor"]["most_frequent_control"]})

[Models/belt_conveyor.py experiment]
  Saved Results\belt_conveyor\belt_conveyor_cora.json, Results\belt_conveyor\belt_conveyor_sensor_visual_comparison.md, and Figures\belt_conveyor\belt_conveyor_cora_confusions.png
Belt Conveyor: matches the saved result


,accuracy,balanced_accuracy,macro_precision,macro_recall,macro_f1
Sensor: T1 temperature,0.625,0.583,0.567,0.583,0.564
Camera: infrared,0.500,0.417,0.433,0.417,0.418
Always guess 'worn',0.750,0.500,0.375,0.500,0.429


![Belt Conveyor sensor vs camera](../Figures/belt_conveyor/belt_conveyor_cora_confusions.png)

**What this means:** weak result, reported honestly. Sensor 0.625 and camera 0.500 are both **below 0.750**, the score from always guessing "worn" (12 of 16 runs are worn). Only 16 runs from one rig. Full write-up: [`Results/belt_conveyor/belt_conveyor_sensor_visual_comparison.md`](../Results/belt_conveyor/belt_conveyor_sensor_visual_comparison.md).

**Data (CC BY 4.0):** CORA v2.1, DOI 10.34810/DATA2500.

## Week 5: baseline model comparison
Four model types are trained with settings fixed in advance and tested on exactly the same held-out recordings as above: a majority-class control, logistic regression, a decision tree and a random forest. No model is picked using these test results; the main model stays the same.

In [3]:
if all(LOCAL.values()):
    run_script("Models/belt_conveyor.py", "baselines", tail=1)
show_baselines(load("belt_conveyor_baselines.json"))

[Models/belt_conveyor.py baselines]
  Saved Results\belt_conveyor\belt_conveyor_baselines.json and Figures\belt_conveyor\belt_conveyor_baselines.png


,accuracy,balanced_accuracy,macro_precision,macro_recall,macro_f1
Sensor: T1 temperature | Majority class (control),0.750,0.500,0.375,0.500,0.429
Sensor: T1 temperature | Logistic regression,0.625,0.583,0.567,0.583,0.564
Sensor: T1 temperature | Decision tree,0.562,0.458,0.464,0.458,0.459
Sensor: T1 temperature | Random forest,0.625,0.417,0.357,0.417,0.385
Camera: infrared | Majority class (control),0.750,0.500,0.375,0.500,0.429
Camera: infrared | Logistic regression,0.500,0.417,0.433,0.417,0.418
Camera: infrared | Decision tree,0.688,0.625,0.609,0.625,0.614
Camera: infrared | Random forest,0.688,0.542,0.551,0.542,0.543


![Baseline comparison](../Figures/belt_conveyor/belt_conveyor_baselines.png)

## Week 6: professor's remarks (4 classes, CSV sensor model, CNN with augmentation)
- **CSV:** the sensor features are written to `Processed_Data/belt_conveyor_sensor_features_4class.csv` and 4-class models are trained on it.
- **Named images:** `export-images` saves the camera images as `Images/belt_conveyor/<class>/belt_conveyor_<class>_..._tNN.png` (local only).
- **CSV model choice:** the feature set and model are chosen on validation data inside the training set only, then tested once.
- **CNN:** a pretrained ResNet18 (transfer learning) and a small CNN, both with data augmentation (flip, shift, brightness, noise), tested on the same held-out recordings. Retraining takes minutes, so it is off by default; set `TRAIN_CNN = True` to retrain.
- **Two tests:** the *main* test holds out whole recordings (or whole speeds/bearings) as before. The *time split* (secondary, easier) trains on the first 60% of every recording and tests on the last 30% (10% gap); the same recordings appear in training, so part of a high score may come from recognising the recording rather than the fault.

In [4]:
TRAIN_CNN = False
if all(LOCAL.values()):
    run_script("Models/belt_conveyor.py", "csv", tail=2)
    run_script("Models/belt_conveyor.py", "csv", "--split", "time", tail=1)
    if TRAIN_CNN:
        run_script("Models/belt_conveyor.py", "export-images", tail=1)
        run_script("Models/belt_conveyor.py", "cnn-pretrained", tail=1)
        run_script("Models/belt_conveyor.py", "cnn-pretrained", "--split", "time", tail=1)
        run_script("Models/belt_conveyor.py", "cnn", tail=1)
show_week6("belt_conveyor")

C:\Users\laksh\Downloads\GitHub\capstone-week4\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Users\laksh\Downloads\GitHub\capstone-week4\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Users\laksh\Downloads\GitHub\capstone-week4\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


C:\Users\laksh\Downloads\GitHub\capstone-week4\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Users\laksh\Downloads\GitHub\capstone-week4\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Users\laksh\Downloads\GitHub\capstone-week4\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


C:\Users\laksh\Downloads\GitHub\capstone-week4\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Users\laksh\Downloads\GitHub\capstone-week4\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


C:\Users\laksh\Downloads\GitHub\capstone-week4\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


C:\Users\laksh\Downloads\GitHub\capstone-week4\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


...

C:\Users\laksh\Downloads\GitHub\capstone-week4\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


C:\Users\laksh\Downloads\GitHub\capstone-week4\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Users\laksh\Downloads\GitHub\capstone-week4\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


[Models/belt_conveyor.py csv]
  Validation-selected model: accuracy 0.625  macro F1 0.625  (choices: ['vibration+temperature (per-speed normalised) / Extra trees', 'vibration+temperature (per-speed normalised) / Logistic regression', 'vibration+temperature (per-speed normalised) / SVM (RBF)', 'vibration+temperature (per-speed normalised) / Logistic regression'])
  Saved Processed_Data\belt_conveyor_sensor_features_4class.csv, Results\belt_conveyor\belt_conveyor_csv_4class.json and Figures\belt_conveyor\belt_conveyor_csv_4class.png


[Models/belt_conveyor.py csv --split time]
  Saved Results\belt_conveyor\belt_conveyor_csv_4class_timesplit.json and Figures\belt_conveyor\belt_conveyor_csv_4class_timesplit.png


,accuracy,balanced_accuracy,macro_precision,macro_recall,macro_f1
CSV sensor | validation-selected (main),0.625,0.625,0.625,0.625,0.625
CSV sensor | Majority class (control),0.250,0.250,0.062,0.250,0.100
CSV sensor | Logistic regression,0.375,0.375,0.333,0.375,0.330
CSV sensor | Random forest,0.375,0.375,0.299,0.375,0.288
"Camera images | pretrained ResNet18 (main), per recording",0.562,0.562,0.562,0.562,0.521
"Camera images | small CNN, per recording",0.312,0.312,0.317,0.312,0.205
TIME SPLIT (secondary) | CSV sensor | Random forest,1.000,1.000,1.000,1.000,1.000
TIME SPLIT (secondary) | Camera images | pretrained ResNet18,1.000,1.000,1.000,1.000,1.000


![CSV sensor model](../Figures/belt_conveyor/belt_conveyor_csv_4class.png) ![Pretrained CNN](../Figures/belt_conveyor/belt_conveyor_cnn_pretrained_4class.png)

Time split (secondary): ![CSV time split](../Figures/belt_conveyor/belt_conveyor_csv_4class_timesplit.png) ![CNN time split](../Figures/belt_conveyor/belt_conveyor_cnn_pretrained_4class_timesplit.png)

## Try your own CSV
Put the path of any CSV in `CSV_FILE` and run this cell. It can be a **feature table** (same columns as `Processed_Data/belt_conveyor_sensor_features_4class.csv`, optional `class` column) or a **raw recording** (one column of sensor values). Set `LABEL` to the true class to see the accuracy. Classes: healthy, wear25, wear50, wear75. The saved model is trained on all project data, so testing on one of the project's own recordings gives an optimistic score; to test fairly, first run `python Models/belt_conveyor.py train-final --exclude <recording>`.

In [5]:
CSV_FILE = None   # e.g. r"C:/Users/me/Downloads/my_recording.csv"
LABEL = None      # true class if known
SPEED = None      # raw recordings only: "F5", "F15", "F50" or "F60"
if CSV_FILE:
    run_script("Models/belt_conveyor.py", "predict", "--csv", CSV_FILE, *((["--label", LABEL] if LABEL else []) + (["--speed", SPEED] if SPEED else [])), tail=25)
else:
    print("Set CSV_FILE to a CSV path to get predictions (and accuracy if LABEL is set).")

Set CSV_FILE to a CSV path to get predictions (and accuracy if LABEL is set).
